In [1]:
import eval_beir_helpers
from redis_bm25 import gather_bm25_results
from redis_vector import gather_res_vector
from beir.retrieval.evaluation import EvaluateRetrieval

/Users/robert.shelton/.pyenv/versions/3.11.9/lib/python3.11/site-packages/beir/util.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm
/Users/robert.shelton/.pyenv/versions/3.11.9/lib/python3.11/site-packages/huggingface_hub/file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/Users/robert.shelton/.pyenv/versions/3.11.9/lib/python3.11/site-packages/huggingface_hub/file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


## Eval beir datasets against Redis with different search techniques

### Load dataset

In [2]:
from redisvl.utils.vectorize import HFTextVectorizer

# set vectorizer
emb_model = HFTextVectorizer("sentence-transformers/all-MiniLM-L6-v2")

In [3]:
# set dataset of interest
dataset = "scifact"

corpus, queries, qrels = eval_beir_helpers.get_beir_dataset(dataset)
corpus_data = eval_beir_helpers.process_corpus(corpus, emb_model=emb_model)

./beir_datasets/scifact.zip:   0%|          | 0.00/2.69M [00:00<?, ?iB/s]

  0%|          | 0/5183 [00:00<?, ?it/s]

In [4]:
index = eval_beir_helpers.create_redis_index(dataset)

In [5]:
# load if need be
index.load(corpus_data)

['scifact:01JQ9C2JPSNRWR5TY0EDPZA8EX',
 'scifact:01JQ9C2JPTMC3PC6C8A4MJPYCS',
 'scifact:01JQ9C2JPTY2KBRXKQDEH1SFF7',
 'scifact:01JQ9C2JPTK72ZA4FCFR9E0PP1',
 'scifact:01JQ9C2JPT5D5KRB7YKXXJ5Y2A',
 'scifact:01JQ9C2JPTX4W8YTQSABBYGP7X',
 'scifact:01JQ9C2JPTKVW5RXG1B5E0651Z',
 'scifact:01JQ9C2JPTE5DVTXBSDMX55J6D',
 'scifact:01JQ9C2JPTZS4G7CK838HYDMGC',
 'scifact:01JQ9C2JPT1V0EYB44EM45FGW8',
 'scifact:01JQ9C2JPT6HPHEBZ9RHKRMVAX',
 'scifact:01JQ9C2JPTDNG50N6T8DFNRFDC',
 'scifact:01JQ9C2JPTGQM8QGEFP5TCNSAA',
 'scifact:01JQ9C2JPT3ADBGRMHAYY6Z8JD',
 'scifact:01JQ9C2JPT6NQR95WJ6FVSRHH9',
 'scifact:01JQ9C2JPTTDHZE342WJJ838SP',
 'scifact:01JQ9C2JPTABWVWTQD9X4QTJV9',
 'scifact:01JQ9C2JPTEEJJSYVMJ29ZGXM3',
 'scifact:01JQ9C2JPT6NR8879TZQN89WXY',
 'scifact:01JQ9C2JPT93N6E0RE5T6H9ZQ8',
 'scifact:01JQ9C2JPTCF6D1GRBNJ5B32AR',
 'scifact:01JQ9C2JPTNGQX3YTMR4A1N5G6',
 'scifact:01JQ9C2JPTF0ZA1Y287HDN8W5J',
 'scifact:01JQ9C2JPTF5SWBV2GTG2HB0ZP',
 'scifact:01JQ9C2JPT7V1AXTB53TYN01E1',
 'scifact:01JQ9C2JPTTEVAQ

### Check index info

See if all docs indexed, if there were any indexing errors, and view the total indexing time.

In [6]:
num_docs = index.info()["num_docs"]
total_indexing_time = round(float(index.info()["total_indexing_time"]) / 1000, 3) # indexing time in ms
indexing_failures = index.info()['Index Errors']

print(f"{num_docs=}, {total_indexing_time=} s, {indexing_failures=}")

num_docs=5183, total_indexing_time=4.826 s, indexing_failures=['indexing failures', 0, 'last indexing error', 'N/A', 'last indexing error key', 'N/A']


## Eval for different search methods

In [7]:
# simple bm25

redis_res_bm25 = gather_bm25_results(queries, index)
redis_bm25_ndcg, _map, redis_bm25_recall, redis_bm25_precision = (
    EvaluateRetrieval.evaluate(qrels, redis_res_bm25, [1, 10])
)

print(f"{redis_bm25_ndcg=} \n {redis_bm25_recall=} \n {redis_bm25_precision=}")

redis_bm25_ndcg={'NDCG@1': 0.47333, 'NDCG@10': 0.58671} 
 redis_bm25_recall={'Recall@1': 0.46167, 'Recall@10': 0.70533} 
 redis_bm25_precision={'P@1': 0.47333, 'P@10': 0.07833}


In [8]:
# basic vector

redis_res_vector = gather_res_vector(queries, index, emb_model)
vec_ndcg, _map, vec_recall, vec_precision = EvaluateRetrieval.evaluate(
    qrels, redis_res_vector, [1, 10]
)

print(f"{vec_ndcg=} \n {vec_recall=} \n {vec_precision=}")

vec_ndcg={'NDCG@1': 0.49667, 'NDCG@10': 0.61986} 
 vec_recall={'Recall@1': 0.47567, 'Recall@10': 0.744} 
 vec_precision={'P@1': 0.49667, 'P@10': 0.085}


In [9]:
# lin combo
from redis_lin_combo import gather_lin_combo_results

alpha = 0.7
redis_res_lin_combo = gather_lin_combo_results(queries, index, alpha, emb_model)
lin_combo_ndcg, _map, lin_combo_recall, lin_combo_precision = EvaluateRetrieval.evaluate(
    qrels, redis_res_lin_combo, [1, 10]
)

print(f"{lin_combo_ndcg=} \n {lin_combo_recall=} \n {lin_combo_precision=}")

lin_combo_ndcg={'NDCG@1': 0.52, 'NDCG@10': 0.64888} 
 lin_combo_recall={'Recall@1': 0.505, 'Recall@10': 0.78333} 
 lin_combo_precision={'P@1': 0.52, 'P@10': 0.08833}


In [10]:
# weighted rrf
from redis_weighted_rrf import gather_weighted_rrf

redis_res_w_rrf = gather_weighted_rrf(queries, index, emb_model)
w_rrf_ndcg, _map, w_rrf_recall, w_rrf_precision = EvaluateRetrieval.evaluate(
    qrels, redis_res_w_rrf, [1, 10]
)

print(f"{w_rrf_ndcg=} \n {w_rrf_recall=} \n {w_rrf_precision=}")

w_rrf_ndcg={'NDCG@1': 0.48667, 'NDCG@10': 0.64578} 
 w_rrf_recall={'Recall@1': 0.46956, 'Recall@10': 0.79667} 
 w_rrf_precision={'P@1': 0.48667, 'P@10': 0.09033}


In [11]:
# weighted rrf
from redis_rerank import gather_rerank_results

redis_res_rerank = gather_rerank_results(queries, index, emb_model)
rerank_ndcg, _map, rerank_recall, rerank_precision = EvaluateRetrieval.evaluate(
    qrels, redis_res_rerank, [1, 10]
)

print(f"{rerank_ndcg=} \n {rerank_recall=} \n {rerank_precision=}")

rerank_ndcg={'NDCG@1': 0.46667, 'NDCG@10': 0.62705} 
 rerank_recall={'Recall@1': 0.44317, 'Recall@10': 0.82122} 
 rerank_precision={'P@1': 0.46667, 'P@10': 0.093}


# Save outputs of run

In [12]:
import pandas as pd

res_data = {
    "Model": ["redis_BM25", "redis_vector", "lin_combo", "weighted_rrf", "rerank"],
    "NDCG@1": [redis_bm25_ndcg['NDCG@1'], vec_ndcg['NDCG@1'], lin_combo_ndcg['NDCG@1'], w_rrf_ndcg['NDCG@1'], rerank_ndcg['NDCG@1']],
    "NDCG@10": [redis_bm25_ndcg['NDCG@10'], vec_ndcg['NDCG@10'], lin_combo_ndcg['NDCG@10'], w_rrf_ndcg['NDCG@10'], rerank_ndcg['NDCG@10']],
    "Recall@1": [redis_bm25_recall['Recall@1'], vec_recall['Recall@1'], lin_combo_recall['Recall@1'], w_rrf_recall['Recall@1'], rerank_recall['Recall@1']],
    "Recall@10": [redis_bm25_recall['Recall@10'], vec_recall['Recall@10'], lin_combo_recall['Recall@10'], w_rrf_recall['Recall@10'], rerank_recall['Recall@10']],
    "Precision@1": [redis_bm25_precision['P@1'], vec_precision['P@1'], lin_combo_precision['P@1'], w_rrf_precision['P@1'], rerank_precision['P@1']],
    "Precision@10": [redis_bm25_precision['P@10'], vec_precision['P@10'], lin_combo_precision['P@10'], w_rrf_precision['P@10'], rerank_precision['P@10']]
}

df = pd.DataFrame(res_data)

In [13]:
df.to_csv(f"{dataset}_results.csv", index=False)

## Cleanup

In [14]:
index.delete()